In [1]:
%pip install gensim

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.6/27.6 MB 149.9 kB/s  0:03:31m0:00:0100:07
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [gensim]2m1/2 [gensim]

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### Section 1: Data Collection, Preprocessing & Dynamic Sequence Batching

In [2]:
import re
import math
import random
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt

import nltk
from nltk.corpus import gutenberg

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence, pad_sequence

from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MAX_LEN = 20
EMBED_DIM = 50
HIDDEN_DIM = 64
PAD_IDX = 0
UNK_IDX = 1

# Where plots get saved. Uses a local "outputs" folder next to this notebook/script
# so it works on any machine (Windows/Mac/Linux, lab or personal laptop).
import os
OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
import gensim.downloader as gensim_api


#### Task 1.1: Data Collection

In [3]:
nltk.download("gutenberg")
nltk.download("punkt")
nltk.download("punkt_tab")
raw_text = gutenberg.raw("carroll-alice.txt")
sentences = nltk.sent_tokenize(raw_text)

[nltk_data] Downloading package gutenberg to /home/user/nltk_data...
[nltk_data]   Unzipping corpora/gutenberg.zip.
[nltk_data] Downloading package punkt to /home/user/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /home/user/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [16]:
print(len(sentences))
print(len(raw_text))

1625
144395


In [11]:
raw_words = nltk.word_tokenize(raw_text)
len(raw_words)

33535

In [18]:
unq_vocab = set(raw_words)
len(unq_vocab)

3157

### Task 1.2: Tokenization, Truncation, and Dynamic Sequence Generation

In [39]:
def transform_raw_sentence(raw_sentence):
    raw_sentence = raw_sentence.lower()
    raw_sentence = re.sub(r"([.,!?])", r" \1 ", raw_sentence)
    raw_sentence = re.sub(r"[^a-z0-9\.,!\?\s]+", "", raw_sentence)
    return raw_sentence.split()

tokenised_sentence = [transform_raw_sentence(sentence) for sentence in sentences]

In [40]:
all_words = [word for tokens in tokenised_sentence for word in tokens]
freq_table = Counter(all_words)
word2idx = {"<pad>" : 0, "<unk" : 1}
for word, count in freq_table.most_common():
    word2idx[word] = len(word2idx)

idx2word = {idx : word for word,idx in word2idx.items()}

In [45]:
def generate_prefix_pairs(token_indexes, max_len=20):
    pairs = []
    for i in range(1, len(token_indexes)):
        context = token_indexes[:i]
        if len(context) > max_len:
            context = context[-max_len:]
        target = token_indexes[i]
        pairs.append((context,target))
    return pairs

all_pairs = []
for sentence_tokens in tokenised_sentence:
    sentence_indexes = [word2idx.get(w,word2idx["<unk>"]) for w in sentence_tokens]
    pairs = generate_prefix_pairs(sentence_indexes, max_len=20)
    all_pairs.extend(pairs)

print(f"Raw text: {sentences[0]}")
print(f"\nWord count: {len(sentence_tokens)}")
print(f"\nCleaned tokens: ")
for idx, (token, id) in enumerate(zip(sentence_tokens, sentence_indexes)):
    print(f"[{idx:02d}] {token:<12} - {id}")

for i, (X,y) in enumerate(pairs[:3]):
    x_words = [idx2word[idx] for idx in X]
    print(f"step {i+1:02d} - X (len {len(X) : 02d}) : {X} - Text: {x_words} - Target (y) : {y} ({idx2word[y]})")

if(len(pairs) >= 22):
    print("Truncation points:")
    for i in range(19,22):
        X,y = pairs[i]
        x_words = [idx2word[idx] for idx in X]
        print(f"step {i+1:02d} - X (len {len(X) : 02d}) : {X} - Target: ({idx2word[y]}) - first context token: {x_words[0]}")

KeyError: '<unk>'

#### Task 1.3: Dynamic Padding and PyTorch DataLoaders

In [46]:
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from sklearn.model_selection import train_test_split

class Textdataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self,idx):
        return self.pairs[idx]

def my_collate_fn(batch):
    sorted_batch = sorted(batch, key=lambda pair: len(pair[0]), reverse=True)
    x_seqs = [torch.tensor(pair[0],dtype=torch.long) for pair in sorted_batch]
    y_targets = [pair[1] for pair in sorted_batch]
    lengths = torch.tensor([len(x) for x in x_seqs], dtype=torch.long)
    X_padded = pad_sequence(x_seqs, batch_first=True, padding_value=0)
    Y = torch.tensor(y_targets, dtype=torch.long)
    return X_padded, Y, lengths

In [47]:
train_pairs, test_pairs = train_test_split(pairs, test_size=0.20, random_state=42)
training_dataset = Textdataset(train_pairs)
testing_dataset = Textdataset(test_pairs)

train_loader = DataLoader(training_dataset, batch_size=128, shuffle=True, collate_fn=my_collate_fn)
test_loader = DataLoader(testing_dataset, batch_size=128, shuffle=False, collate_fn=my_collate_fn)

X_batch, Y_batch, batch_lengths = next(iter(train_loader))
print(f"X_padded shape : {X_batch.shape}")
print(f"Y shape : {Y_batch.shape}")
print(f"Lengths shape : {batch_lengths.shape}")

ValueError: too many dimensions 'str'

### Section 2: Model Architectures, Embedding Experiments, and Training

#### Task 2.1: Baseline Model — Vanilla RNN with Frozen GloVe

In [50]:
class BaselineRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim=50, hidden_dim=64, num_classes=2):
        super(BaselineRNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.embedding.weight.self.requires_grad = False
        self.rnn = nn.RNN(input=embed_dim, hidden_size=hidden_dim,batch_first=True)
        self.dropout = nn.Dropout(0.2)
        self.fc = nn.Linear(hidden_dim, num_classes)
    def load_glove_weights(self,glove_weights_tensor):
        self.embedding.weight.data.copy_(glove_weights_tensor)
        self.embedding.weight.requires_grad = False
    def forward(self, input_ids, lengths):
        embedded = self.embedding(input_ids)
        packed_embedded = pack_padded_sequence(embedded, lengths, batch_first=True, enforce_sorted=True)
        packed_output, h_n = self.rnn(packed_embedded)
        last_hidden = h_n[-1]
        out = self.dropout(last_hidden)
        logits = self.fc(out)
        return logits
    def count_params(model):
        trainable_params = sum(p.numel() for p in model.parameters if p.requires_grad)
        non_trainable_params = sum(p.numel() for p in model.parameters if not p.requires_grad)
        total_params = trainable_params + non_trainable_params
        print(f"Trainable Parameters : {trainable_params}")
        print(f"Non-Trainable Parameters : {non_trainable_params}")
        print(f"Total Parameters : {total_params}")

#### Task 2.2: Generic Training & Evaluation Pipeline

In [ ]:
import math
import matplotlib.pyplot as plt
import torch.nn as nn
import torch.optim as optim

def train_model(model, model_name, train_loader, test_loader, epochs=10, lr=0.002):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    criterion = nn.CrossEntropyLoss(ignore_index=0)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    history = {"loss" : [], "accuracy" : [], "ppl" : []}
    print("Model training :")
    for epoch in range(1, epochs+1):
        model.train()
        curr_train_loss = 0.0
        for batch in train_loader:
            if len(batch) == 3:
                inputs, lengths, targets = batch
                inputs, lengths, targets = (inputs.to(device), lengths.to(device), targets.to(device))
                optimizer.zero_grad()
                outputs = model(inputs, lengths)
            else:
                inputs, targets = batch
                inputs, targets = inputs.to(device), targets.to(device)
                optimizer.zero_grad()
                outputs = model(inputs, lengths)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            curr_train_loss += loss
        model.eval()
        curr